In [1]:
import os
import shutil
from sklearn.model_selection import train_test_split
from pathlib import Path

# Конфигурация (повторяем ваш код)
dataset_path = "dataset/train"
train_dest = "dataset/train_split"
val_dest = "dataset/val_split"
val_ratio = 0.2
random_seed = 42

Path(train_dest).mkdir(parents=True, exist_ok=True)
Path(val_dest).mkdir(parents=True, exist_ok=True)

all_files = [f for f in os.listdir(dataset_path) if f.endswith(('.jpg', '.png', '.jpeg'))]
train_files, val_files = train_test_split(all_files, test_size=val_ratio, random_state=random_seed)

def copy_files(files, destination):
    for file in files:
        base_name = os.path.splitext(file)[0]
        shutil.copy(os.path.join(dataset_path, file), os.path.join(destination, file))
        txt_file = f"{base_name}.txt"
        txt_src = os.path.join(dataset_path, txt_file)
        if os.path.exists(txt_src):
            shutil.copy(txt_src, os.path.join(destination, txt_file))

copy_files(train_files, train_dest)
copy_files(val_files, val_dest)

In [3]:
import torch
from ultralytics import YOLO
from pathlib import Path
import numpy as np
from sklearn.metrics import precision_recall_curve, average_precision_score

# Загрузка модели
model = YOLO("runs/result/preprocessdetect122spring.pt")

# Валидация (встроенная функция Ultralytics)
results = model.val(data="data-ul.yaml", imgsz=640, batch=16, conf=0.25, iou=0.5)
print(f"mAP@0.5: {results.box.map50:.4f}")
print(f"mAP@0.5:0.95: {results.box.map:.4f}")
print(f"Precision: {results.box.mp:.4f}")
print(f"Recall: {results.box.mr:.4f}")

# Если нужен свой цикл по батчам для визуализации
val_images = list(Path(val_dest).glob("*.jpg")) + list(Path(val_dest).glob("*.png"))
batch_size = 32
for i in range(0, len(val_images), batch_size):
    batch = val_images[i:i+batch_size]
    # Прогон модели
    preds = model([str(p) for p in batch], imgsz=640, conf=0.25, iou=0.5)
    # Обработка предсказаний...

Ultralytics 8.3.250  Python-3.14.2 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
val: Fast image access  (ping: 0.40.1 ms, read: 146.623.9 MB/s, size: 1211.4 KB)
val: Scanning D:\AIM\AI-Flow-Detecting\ai-core\dataset\val_split_ul.cache... 84 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 84/84 53.5Kit/s 0.0s
val: D:\AIM\AI-Flow-Detecting\ai-core\dataset\val_split_ul\20250810_153130.jpg: 2 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.4s/it 8.4s1.1ss
                   all         84        516      0.808      0.703      0.785      0.433
Speed: 3.2ms preprocess, 50.7ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to D:\AIM\AI-Flow-Detecting\runs\detect\val4
mAP@0.5: 0.7848
mAP@0.5:0.95: 0.4332
Precision: 0.8077
Recall: 0.7035

0: 640x640 3 humans, 96.9ms
1: 640x640 9 humans, 96.9ms
2: 640x640 5 humans, 96.9ms
3: 640x640 3 humans, 96.9ms
4: 640x64

In [6]:
import matplotlib.pyplot as plt

# Из результатов валидации можно построить кривую
# Если у вас есть файл results.csv из тренировки
import pandas as pd
df = pd.read_csv("runs/train/yolov_finetuned/results.csv")
plt.plot(df['epoch'], df['metrics/precision(B)'], label='Precision')
plt.plot(df['epoch'], df['metrics/recall(B)'], label='Recall')
plt.xlabel('Epoch')
plt.legend()
plt.title('Precision-Confidence Curve (аналог)')
plt.savefig("precision_curve.png")